In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import evaluate
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import BlipProcessor, BlipForConditionalGeneration

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
ROOT_DIR = "../data_ready_for_kaggle"
TEST_PATH = os.path.join(ROOT_DIR, "cleaned_test.csv")
IMAGES_DIR = os.path.join(ROOT_DIR, "images_resized")
MODEL_DIR = os.path.abspath("../saved_models_v3")
ONNX_FP32_PATH = './model_quantized/onnx_fp32/blip_captioning_fp32.onnx'
SAVE_DIR = './model_quantized/onnx_int8'
RESULTS_DIR = './results'
MAX_TEST_IMAGES = 200
MAX_LENGTH = 64
BATCH_SIZE = 16
NUM_WORKERS = 0
NUM_BEAMS = 5
NO_REPEAT_NGRAM_SIZE = 3
REPETITION_PENALTY = 1.5
WARMUP_STEPS = 10
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
ONNX_VISION_FP32_PATH = os.path.join(os.path.dirname(ONNX_FP32_PATH), 'vision_encoder_fp32.onnx')
ONNX_DECODER_INIT_FP32_PATH = os.path.join(os.path.dirname(ONNX_FP32_PATH), 'text_decoder_init_fp32.onnx')
ONNX_DECODER_WITH_PAST_FP32_PATH = os.path.join(os.path.dirname(ONNX_FP32_PATH), 'text_decoder_with_past_fp32.onnx')

ONNX_VISION_INT8_PATH = os.path.join(SAVE_DIR, 'vision_encoder_int8.onnx')
ONNX_DECODER_INIT_INT8_PATH = os.path.join(SAVE_DIR, 'text_decoder_init_int8.onnx')
ONNX_DECODER_WITH_PAST_INT8_PATH = os.path.join(SAVE_DIR, 'text_decoder_with_past_int8.onnx')

quantize_dynamic(ONNX_VISION_FP32_PATH, ONNX_VISION_INT8_PATH, weight_type=QuantType.QInt8)
quantize_dynamic(ONNX_DECODER_INIT_FP32_PATH, ONNX_DECODER_INIT_INT8_PATH, weight_type=QuantType.QInt8)
quantize_dynamic(ONNX_DECODER_WITH_PAST_FP32_PATH, ONNX_DECODER_WITH_PAST_INT8_PATH, weight_type=QuantType.QInt8)

processor = BlipProcessor.from_pretrained(MODEL_DIR)
model = BlipForConditionalGeneration.from_pretrained(MODEL_DIR)
model.eval()

eos_token_id = model.config.text_config.eos_token_id
pad_token_id = model.config.text_config.pad_token_id
if pad_token_id is None:
    pad_token_id = eos_token_id

num_layers = model.text_decoder.config.num_hidden_layers
encoder_seq_len = model.vision_model.config.image_size // model.vision_model.config.patch_size
encoder_seq_len = encoder_seq_len * encoder_seq_len + 1

class UITViICDataset(Dataset):
    def __init__(self, data_file, img_dir, processor, max_length=64):
        self.df = pd.read_csv(data_file)
        self.img_dir = img_dir
        self.processor = processor
        self.max_length = max_length
        self.image_groups = self.df.groupby('image_file')['caption'].apply(list).reset_index()

    def __len__(self):
        return len(self.image_groups)

    def __getitem__(self, index):
        row = self.image_groups.iloc[index]
        image_file = row['image_file']
        caption = row['caption'][0]
        image = Image.open(os.path.join(self.img_dir, image_file)).convert('RGB')

        encoding = self.processor(
            images=image,
            text=caption,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding['image_file'] = image_file
        encoding['raw_captions'] = row['caption']
        return encoding

def collate_fn(batch):
    result = {}
    for key in ['pixel_values', 'input_ids', 'attention_mask']:
        result[key] = torch.stack([item[key] for item in batch])
    result['image_file'] = [item['image_file'] for item in batch]
    result['raw_captions'] = [item['raw_captions'] for item in batch]
    return result

def build_multi_references(dataloader):
    references = {}
    for batch in dataloader:
        for image_file, raw_captions in zip(batch['image_file'], batch['raw_captions']):
            references[image_file] = raw_captions
    return references

def calculate_metrics(predictions_dict, references_dict):
    common_images = sorted(set(predictions_dict.keys()) & set(references_dict.keys()))
    predictions = [predictions_dict[img] for img in common_images]
    references = [references_dict[img] for img in common_images]

    bleu_metric = evaluate.load('bleu')
    rouge_metric = evaluate.load('rouge')
    meteor_metric = evaluate.load('meteor')

    bleu_results = bleu_metric.compute(predictions=predictions, references=references)
    rouge_results = rouge_metric.compute(predictions=predictions, references=references)
    meteor_results = meteor_metric.compute(predictions=predictions, references=references)

    return {
        'bleu': bleu_results,
        'rouge': rouge_results,
        'meteor': meteor_results,
        'num_images': len(common_images),
    }

def summarize_benchmark(name, predictions_dict, references_dict, latencies, backend, precision, device, provider=None, generation_strategy=None, cache_enabled=None):
    metrics = calculate_metrics(predictions_dict, references_dict)

    avg_latency = float(np.mean(latencies)) if latencies else 0.0
    p95_latency = float(np.percentile(latencies, 95)) if latencies else 0.0
    throughput = float(BATCH_SIZE / avg_latency) if avg_latency > 0 else 0.0

    benchmark = {
        'name': name,
        'backend': backend,
        'precision': precision,
        'device': device,
        'provider': provider,
        'generation_strategy': generation_strategy,
        'cache_enabled': cache_enabled,
        'batch_size': BATCH_SIZE,
        'max_test_images': MAX_TEST_IMAGES,
        'avg_latency_seconds_per_batch': avg_latency,
        'p95_latency_seconds_per_batch': p95_latency,
        'throughput_images_per_second': throughput,
        'bleu4': metrics['bleu']['bleu'] * 100,
        'rougeL': metrics['rouge']['rougeL'] * 100,
        'meteor': metrics['meteor']['meteor'] * 100,
        'num_images': metrics['num_images'],
    }
    return benchmark, metrics

def select_provider():
    available = ort.get_available_providers()
    if 'CUDAExecutionProvider' in available:
        return ['CUDAExecutionProvider'], 'cuda', 'CUDAExecutionProvider'
    return ['CPUExecutionProvider'], 'cpu', 'CPUExecutionProvider'

def build_encoder_attention_mask(batch_size):
    return np.ones((batch_size, encoder_seq_len), dtype=np.int64)

def generate_captions_with_onnx(encoder_session, decoder_init_session, decoder_with_past_session, processor, dataloader):
    results = {}
    latencies = []

    for batch_idx, batch in enumerate(tqdm(dataloader, desc='Generating ONNX INT8 captions')):
        batch_pixel_values = batch['pixel_values'].numpy().astype(np.float32)
        batch_size = batch_pixel_values.shape[0]
        encoder_hidden = encoder_session.run(None, {'pixel_values': batch_pixel_values})[0]
        encoder_mask = build_encoder_attention_mask(batch_size)
        generated = np.full((batch_size, 1), model.config.text_config.bos_token_id, dtype=np.int64)
        finished = np.zeros(batch_size, dtype=bool)

        start_time = time.perf_counter()
        init_outputs = decoder_init_session.run(None, {
            'input_ids': generated,
            'attention_mask': np.ones_like(generated, dtype=np.int64),
            'encoder_hidden_states': encoder_hidden.astype(np.float32),
            'encoder_attention_mask': encoder_mask,
        })
        logits = init_outputs[0]
        past = list(init_outputs[1:])
        next_token = logits[:, -1, :].argmax(axis=-1).astype(np.int64)
        next_token = np.where(finished, pad_token_id, next_token)
        generated = np.concatenate([generated, next_token[:, None]], axis=1)
        finished = finished | (next_token == eos_token_id)

        for _ in range(MAX_LENGTH - 2):
            if finished.all():
                break
            decoder_inputs = {
                'input_ids': next_token[:, None],
                'attention_mask': np.ones((batch_size, generated.shape[1]), dtype=np.int64),
                'encoder_hidden_states': encoder_hidden.astype(np.float32),
                'encoder_attention_mask': encoder_mask,
            }
            for layer_idx in range(num_layers):
                offset = layer_idx * 4
                decoder_inputs[f'past_self_key_{layer_idx}'] = past[offset]
                decoder_inputs[f'past_self_value_{layer_idx}'] = past[offset + 1]
                decoder_inputs[f'past_cross_key_{layer_idx}'] = past[offset + 2]
                decoder_inputs[f'past_cross_value_{layer_idx}'] = past[offset + 3]
            step_outputs = decoder_with_past_session.run(None, decoder_inputs)
            logits = step_outputs[0]
            past = list(step_outputs[1:])
            next_token = logits[:, -1, :].argmax(axis=-1).astype(np.int64)
            next_token = np.where(finished, pad_token_id, next_token)
            generated = np.concatenate([generated, next_token[:, None]], axis=1)
            finished = finished | (next_token == eos_token_id)
        end_time = time.perf_counter()

        if batch_idx >= WARMUP_STEPS:
            latencies.append(end_time - start_time)

        preds = processor.batch_decode(generated, skip_special_tokens=True)
        for image_file, pred in zip(batch['image_file'], preds):
            results[image_file] = pred.strip()

    return results, latencies

provider_list, benchmark_device, provider_name = select_provider()
encoder_session = ort.InferenceSession(ONNX_VISION_INT8_PATH, providers=provider_list)
decoder_init_session = ort.InferenceSession(ONNX_DECODER_INIT_INT8_PATH, providers=provider_list)
decoder_with_past_session = ort.InferenceSession(ONNX_DECODER_WITH_PAST_INT8_PATH, providers=provider_list)

test_dataset = UITViICDataset(TEST_PATH, IMAGES_DIR, processor, max_length=MAX_LENGTH)
test_dataset.image_groups = test_dataset.image_groups.head(MAX_TEST_IMAGES).reset_index(drop=True)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
)

references_dict = build_multi_references(test_dataloader)
predictions_dict, latencies = generate_captions_with_onnx(
    encoder_session,
    decoder_init_session,
    decoder_with_past_session,
    processor,
    test_dataloader,
)
benchmark, metrics = summarize_benchmark(
    'onnx_int8',
    predictions_dict,
    references_dict,
    latencies,
    backend='onnx',
    precision='int8',
    device=benchmark_device,
    provider=provider_name,
    generation_strategy='greedy',
    cache_enabled=True,
)
benchmark['model_path'] = {
    'vision_encoder': ONNX_VISION_INT8_PATH,
    'decoder_init': ONNX_DECODER_INIT_INT8_PATH,
    'decoder_with_past': ONNX_DECODER_WITH_PAST_INT8_PATH,
}

quant_info = {
    'onnx_vision_fp32_path': ONNX_VISION_FP32_PATH,
    'onnx_decoder_init_fp32_path': ONNX_DECODER_INIT_FP32_PATH,
    'onnx_decoder_with_past_fp32_path': ONNX_DECODER_WITH_PAST_FP32_PATH,
    'onnx_vision_int8_path': ONNX_VISION_INT8_PATH,
    'onnx_decoder_init_int8_path': ONNX_DECODER_INIT_INT8_PATH,
    'onnx_decoder_with_past_int8_path': ONNX_DECODER_WITH_PAST_INT8_PATH,
}

print(json.dumps(quant_info, indent=2))
print(json.dumps(benchmark, indent=2, ensure_ascii=False))

with open(os.path.join(RESULTS_DIR, 'onnx_int8_quantization.json'), 'w', encoding='utf-8') as f:
    json.dump(quant_info, f, ensure_ascii=False, indent=2)

with open(os.path.join(RESULTS_DIR, 'onnx_int8.json'), 'w', encoding='utf-8') as f:
    json.dump(benchmark, f, ensure_ascii=False, indent=2)